In [6]:
# Install PySpark in Google Colab
# !pip install pyspark dask pandas

import pandas as pd
import dask.dataframe as dd
from google.colab import files

# Upload your files
uploaded = files.upload()

Saving manufacturing_production_120k.csv to manufacturing_production_120k.csv


In [7]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("ManufacturingAnalysis").getOrCreate()

In [8]:
df = spark.read.csv("manufacturing_production_120k.csv", header=True, inferSchema=True)

In [ ]:
df.show(20)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|           993|             30|              34|    388.11|         61.4|          3.88|        Rework|              No|    Charan|
|    PR0000002|     

In [ ]:
df.printSchema()

root
 |-- production_id: string (nullable = true)
 |-- production_date: date (nullable = true)
 |-- plant_id: string (nullable = true)
 |-- machine_id: string (nullable = true)
 |-- operator_id: string (nullable = true)
 |-- shift: string (nullable = true)
 |-- product_code: string (nullable = true)
 |-- product_family: string (nullable = true)
 |-- units_planned: integer (nullable = true)
 |-- units_produced: integer (nullable = true)
 |-- defective_units: integer (nullable = true)
 |-- downtime_minutes: integer (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- temperature_c: double (nullable = true)
 |-- vibration_mm_s: double (nullable = true)
 |-- quality_status: string (nullable = true)
 |-- maintenance_flag: string (nullable = true)
 |-- supervisor: string (nullable = true)



In [ ]:
print("Column Names:", df.columns)

Column Names: ['production_id', 'production_date', 'plant_id', 'machine_id', 'operator_id', 'shift', 'product_code', 'product_family', 'units_planned', 'units_produced', 'defective_units', 'downtime_minutes', 'energy_kwh', 'temperature_c', 'vibration_mm_s', 'quality_status', 'maintenance_flag', 'supervisor']


In [ ]:
print("Total Records:", df.count())

Total Records: 120000


In [ ]:
df.select("production_id", "plant_id", "machine_id", "product_family", "units_produced").show()

+-------------+--------+----------+--------------+--------------+
|production_id|plant_id|machine_id|product_family|units_produced|
+-------------+--------+----------+--------------+--------------+
|    PR0000001|    PL03|     MC011|        Motors|           993|
|    PR0000002|    PL04|     MC019|   Controllers|           836|
|    PR0000003|    PL01|     MC002|        Motors|           380|
|    PR0000004|    PL03|     MC013|         Pumps|           720|
|    PR0000005|    PL04|     MC020|         Pumps|           296|
|    PR0000006|    PL01|     MC002|        Motors|           795|
|    PR0000007|    PL02|     MC007|        Valves|           955|
|    PR0000008|    PL02|     MC009|        Valves|           895|
|    PR0000009|    PL03|     MC015|   Controllers|           289|
|    PR0000010|    PL02|     MC006|   Controllers|           527|
|    PR0000011|    PL03|     MC012|         Pumps|           512|
|    PR0000012|    PL06|     MC030|   Controllers|           968|
|    PR000

In [ ]:
print("Data Types:", df.dtypes)

Data Types: [('production_id', 'string'), ('production_date', 'date'), ('plant_id', 'string'), ('machine_id', 'string'), ('operator_id', 'string'), ('shift', 'string'), ('product_code', 'string'), ('product_family', 'string'), ('units_planned', 'int'), ('units_produced', 'int'), ('defective_units', 'int'), ('downtime_minutes', 'int'), ('energy_kwh', 'double'), ('temperature_c', 'double'), ('vibration_mm_s', 'double'), ('quality_status', 'string'), ('maintenance_flag', 'string'), ('supervisor', 'string')]


In [ ]:
df.sample(withReplacement=False, fraction=0.01, seed=42).limit(10).show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0000256|     2026-02-17|    PL03|     MC012|     OP0477|Morning|     VLV-120|        Valves|          949|           790|             14|               0|    372.68|        58.34|          3.93|          Pass|              No|     Farah|
|    PR0000314|     

Creating dataframes

In [9]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql.functions import lit

In [10]:
data = [
    (101, "Plant Alpha", "Mumbai"),
    (102, "Plant Beta", "Hyderabad"),
    (103, "Plant Gamma", "Chennai")
]
columns = ["plant_id", "plant_name", "city"]

plant_df_1 = spark.createDataFrame(data, columns)

In [11]:
schema = StructType([
    StructField("plant_id", IntegerType(), True),
    StructField("plant_name", StringType(), True),
    StructField("city", StringType(), True)
])

plant_df_2 = spark.createDataFrame(data, schema)

In [ ]:
plant_df_2.printSchema()

root
 |-- plant_id: integer (nullable = true)
 |-- plant_name: string (nullable = true)
 |-- city: string (nullable = true)



In [ ]:
plant_df_final = plant_df_2.withColumn("country", lit("India"))
plant_df_final.show()

+--------+-----------+---------+-------+
|plant_id| plant_name|     city|country|
+--------+-----------+---------+-------+
|     101|Plant Alpha|   Mumbai|  India|
|     102| Plant Beta|Hyderabad|  India|
|     103|Plant Gamma|  Chennai|  India|
+--------+-----------+---------+-------+



pandas dataframe

In [19]:
import pandas as pd
import pyspark.sql.functions

In [13]:
pdf = pd.read_csv("manufacturing_production_120k.csv")

In [14]:
print("First 5 rows:\n", pdf.head(5))

First 5 rows:
   production_id production_date plant_id machine_id operator_id    shift  \
0     PR0000001      2026-08-07     PL03      MC011      OP0275  Evening   
1     PR0000002      2026-08-03     PL04      MC019      OP0489  Morning   
2     PR0000003      2026-05-20     PL01      MC002      OP0189  Evening   
3     PR0000004      2026-01-15     PL03      MC013      OP0076  Morning   
4     PR0000005      2026-09-21     PL04      MC020      OP0501  Morning   

  product_code product_family  units_planned  units_produced  defective_units  \
0      MTR-100         Motors           1060             993               30   
1      CTL-330    Controllers            809             836                0   
2      MTR-300         Motors            442             380               17   
3      PMP-210          Pumps            787             720                7   
4      PMP-110          Pumps            344             296                7   

   downtime_minutes  energy_kwh  temperat

In [ ]:
print("Last 5 rows:\n", pdf.tail(5))

Last 5 rows:
        production_id production_date plant_id machine_id operator_id    shift  \
119995     PR0119996      2026-07-13     PL05      MC022      OP0193  Evening   
119996     PR0119997      2026-04-26     PL05      MC024      OP0124  Evening   
119997     PR0119998      2026-02-01     PL06      MC027      OP0051  Evening   
119998     PR0119999      2026-01-17     PL01      MC001      OP0452  Evening   
119999     PR0120000      2026-08-16     PL02      MC007      OP0420  Morning   

       product_code product_family  units_planned  units_produced  \
119995      MTR-200         Motors            435             399   
119996      CTL-130    Controllers            813             777   
119997      VLV-120         Valves            650             597   
119998      MTR-300         Motors            887             899   
119999      MTR-200         Motors            547             477   

        defective_units  downtime_minutes  energy_kwh  temperature_c  \
119995      

In [ ]:
print("Shape:", pdf.shape)

Shape: (120000, 18)


In [ ]:
print("Columns:", pdf.columns.tolist())

Columns: ['production_id', 'production_date', 'plant_id', 'machine_id', 'operator_id', 'shift', 'product_code', 'product_family', 'units_planned', 'units_produced', 'defective_units', 'downtime_minutes', 'energy_kwh', 'temperature_c', 'vibration_mm_s', 'quality_status', 'maintenance_flag', 'supervisor']


In [ ]:
print("Data Types:\n", pdf.dtypes)

Data Types:
 production_id        object
production_date      object
plant_id             object
machine_id           object
operator_id          object
shift                object
product_code         object
product_family       object
units_planned         int64
units_produced        int64
defective_units       int64
downtime_minutes    float64
energy_kwh          float64
temperature_c       float64
vibration_mm_s      float64
quality_status       object
maintenance_flag     object
supervisor           object
dtype: object


In [21]:
# In Pandas, select columns using double brackets and display using .head()
pdf[["plant_id", "machine_id", "units_produced"]].head()

,plant_id,machine_id,units_produced
0,PL03,MC011,993
1,PL04,MC019,836
2,PL01,MC002,380
3,PL03,MC013,720
4,PL04,MC020,296


In [25]:
pdf_renamed = pdf.rename(columns={"units_produced": "production_output"})

In [26]:
pdf[pdf["quality_status"] == "Reject"]

,production_id,production_date,plant_id,machine_id,operator_id,shift,product_code,product_family,units_planned,units_produced,defective_units,downtime_minutes,energy_kwh,temperature_c,vibration_mm_s,quality_status,maintenance_flag,supervisor
1,PR0000002,2026-08-03,PL04,MC019,OP0489,Morning,CTL-330,Controllers,809,836,0,0.0,398.58,64.37,2.28,Reject,No,Farah
2,PR0000003,2026-05-20,PL01,MC002,OP0189,Evening,MTR-300,Motors,442,380,17,15.0,183.34,54.13,4.66,Reject,No,Deepa
7,PR0000008,2026-08-17,PL02,MC009,OP0106,Night,VLV-320,Valves,862,895,51,77.0,266.06,57.10,1.34,Reject,No,Irfan
9,PR0000010,2026-02-13,PL02,MC006,OP0593,Evening,CTL-230,Controllers,532,527,26,64.0,112.17,60.96,3.48,Reject,No,Gopal
21,PR0000022,2026-03-03,PL02,MC009,OP0294,Evening,MTR-300,Motors,623,560,23,14.0,217.69,55.23,3.28,Reject,Yes,Farah
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119979,PR0119980,2026-07-21,PL02,MC006,OP0591,Morning,VLV-320,Valves,1149,1036,48,40.0,316.55,70.09,2.90,Reject,No,Harini
119988,PR0119989,2026-01-29,PL01,MC003,OP0021,Night,PMP-310,Pumps,1195,1053,26,72.0,240.94,62.27,4.93,Reject,No,Farah
119989,PR0119990,2026-07-20,PL06,MC027,OP0217,Evening,CTL-230,Controllers,782,796,10,71.0,150.47,66.59,4.08,Reject,No,Bharat
119991,PR0119992,2026-04-07,PL05,MC024,OP0186,Morning,PMP-310,Pumps,1187,1024,35,32.0,486.71,54.09,3.37,Reject,No,Farah


In [28]:
pdf[pdf["temperature_c"] > 75]

,production_id,production_date,plant_id,machine_id,operator_id,shift,product_code,product_family,units_planned,units_produced,defective_units,downtime_minutes,energy_kwh,temperature_c,vibration_mm_s,quality_status,maintenance_flag,supervisor
14,PR0000015,2026-03-15,PL02,MC007,OP0238,Morning,MTR-300,Motors,1086,1042,4,172.0,239.64,76.39,2.81,Pass,No,Deepa
16,PR0000017,2026-08-04,PL03,MC011,OP0533,Morning,CTL-330,Controllers,1152,963,28,91.0,347.34,75.09,4.00,Pass,Yes,Gopal
19,PR0000020,2026-02-18,PL03,MC014,OP0563,Morning,PMP-310,Pumps,646,573,13,64.0,277.73,76.92,2.35,Pass,No,Irfan
26,PR0000027,2026-01-03,PL05,MC023,OP0190,Morning,PMP-210,Pumps,358,310,6,46.0,78.29,75.42,3.72,Rework,No,Anita
54,PR0000055,2026-07-31,PL01,MC005,OP0212,Morning,CTL-230,Controllers,499,478,10,32.0,112.60,75.10,3.38,Pass,No,Farah
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119767,PR0119768,2026-01-29,PL02,MC007,OP0262,Evening,CTL-130,Controllers,1129,1139,0,30.0,362.12,77.17,1.98,Pass,No,Anita
119810,PR0119811,2026-07-09,PL05,MC024,OP0372,Evening,CTL-230,Controllers,819,687,45,50.0,171.84,75.24,4.17,Reject,No,Bharat
119833,PR0119834,2026-06-22,PL04,MC020,OP0038,Morning,CTL-330,Controllers,749,669,12,46.0,291.22,80.53,3.05,Pass,No,Anita
119848,PR0119849,2026-02-25,PL06,MC027,OP0563,Morning,PMP-310,Pumps,925,816,28,0.0,258.03,75.94,3.53,Rework,No,Charan


dask dataframe

In [30]:
import dask.dataframe as dd

ddf = dd.read_csv("manufacturing_production_120k.csv")

In [32]:
# Explicitly define the dtype for 'downtime_minutes' to resolve Dask's inference mismatch
ddf = dd.read_csv("manufacturing_production_120k.csv", dtype={"downtime_minutes": "float64"})
print(ddf.head())

  production_id production_date plant_id machine_id operator_id    shift  \
0     PR0000001      2026-08-07     PL03      MC011      OP0275  Evening   
1     PR0000002      2026-08-03     PL04      MC019      OP0489  Morning   
2     PR0000003      2026-05-20     PL01      MC002      OP0189  Evening   
3     PR0000004      2026-01-15     PL03      MC013      OP0076  Morning   
4     PR0000005      2026-09-21     PL04      MC020      OP0501  Morning   

  product_code product_family  units_planned  units_produced  defective_units  \
0      MTR-100         Motors           1060             993               30   
1      CTL-330    Controllers            809             836                0   
2      MTR-300         Motors            442             380               17   
3      PMP-210          Pumps            787             720                7   
4      PMP-110          Pumps            344             296                7   

   downtime_minutes  energy_kwh  temperature_c  vibratio

In [33]:
print("Partitions:", ddf.npartitions)

Partitions: 1


In [34]:
ddf[["plant_id", "product_family", "units_produced"]]

,plant_id,product_family,units_produced
npartitions=1,,,
,string,string,int64
,...,...,...


In [38]:
ddf[ddf["quality_status"] == "Reject"]

,production_id,production_date,plant_id,machine_id,operator_id,shift,product_code,product_family,units_planned,units_produced,defective_units,downtime_minutes,energy_kwh,temperature_c,vibration_mm_s,quality_status,maintenance_flag,supervisor
npartitions=1,,,,,,,,,,,,,,,,,,
,string,string,string,string,string,string,string,string,int64,int64,int64,float64,float64,float64,float64,string,string,string
,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...


In [45]:
avg_units = ddf.groupby("plant_id")["units_produced"].mean()

In [46]:
total_defects =ddf.groupby("product_family")["defective_units"].sum()

In [47]:
print(avg_units.compute())
print(total_defects.compute())

plant_id
PL01    672.998304
PL02    672.781326
PL03    675.665503
PL04    673.835101
PL05    673.124994
PL06    678.006884
Name: units_produced, dtype: float64
product_family
Controllers    507781
Motors         505287
Pumps          507864
Valves         509312
Name: defective_units, dtype: int64


manipulating dataframes


In [48]:
from pyspark.sql.functions import col, lit

In [51]:
dft = df \
    .withColumn("production_efficiency", (col("units_produced") / col("units_planned")) * 100) \
    .withColumn("good_units", col("units_produced") - col("defective_units")) \
    .withColumn("country", lit("India"))

In [52]:
dft.show(5)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------------+----------+-------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|production_efficiency|good_units|country|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------------+----------+-------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|           993|             3

In [57]:
dft=df.withColumnRenamed("energy_kwh", "energy_consumed")
dft.show(5)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+---------------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_consumed|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+---------------+-------------+--------------+--------------+----------------+----------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|           993|             30|              34|         388.11|         61.4|          3.88|        Rework|              No|    Charan|


In [66]:
dft.withColumn("units_produced", col("units_produced").cast("double"))

DataFrame[production_id: string, production_date: date, plant_id: string, machine_id: string, operator_id: string, shift: string, product_code: string, product_family: string, units_planned: int, units_produced: double, defective_units: int, downtime_minutes: int, energy_consumed: double, temperature_c: double, vibration_mm_s: double, quality_status: string, maintenance_flag: string, supervisor: string]

In [67]:
dft.withColumn("defect_percentage", (col("defective_units") / col("units_produced")) * 100)

DataFrame[production_id: string, production_date: date, plant_id: string, machine_id: string, operator_id: string, shift: string, product_code: string, product_family: string, units_planned: int, units_produced: int, defective_units: int, downtime_minutes: int, energy_consumed: double, temperature_c: double, vibration_mm_s: double, quality_status: string, maintenance_flag: string, supervisor: string, defect_percentage: double]

In [68]:
dft.withColumn("energy_per_unit", col("energy_consumed") / col("units_produced"))

DataFrame[production_id: string, production_date: date, plant_id: string, machine_id: string, operator_id: string, shift: string, product_code: string, product_family: string, units_planned: int, units_produced: int, defective_units: int, downtime_minutes: int, energy_consumed: double, temperature_c: double, vibration_mm_s: double, quality_status: string, maintenance_flag: string, supervisor: string, energy_per_unit: double]

In [70]:
df_transformed = df \
    .withColumn("production_efficiency", (col("units_produced") / col("units_planned")) * 100) \
    .withColumn("good_units", col("units_produced") - col("defective_units")) \
    .withColumn("country", lit("India")) \
    .withColumnRenamed("energy_kwh", "energy_consumed") \
    .withColumn("units_produced", col("units_produced").cast("double")) \
    .withColumn("defect_percentage", (col("defective_units") / col("units_produced")) * 100) \
    .withColumn("energy_per_unit", col("energy_consumed") / col("units_produced"))

df_transformed.select("production_id", "production_efficiency", "good_units", "country", "defect_percentage", "energy_per_unit").show(10)

+-------------+---------------------+----------+-------+------------------+-------------------+
|production_id|production_efficiency|good_units|country| defect_percentage|    energy_per_unit|
+-------------+---------------------+----------+-------+------------------+-------------------+
|    PR0000001|    93.67924528301886|       963|  India|3.0211480362537766|0.39084592145015107|
|    PR0000002|   103.33745364647713|       836|  India|               0.0| 0.4767703349282296|
|    PR0000003|    85.97285067873304|       363|  India| 4.473684210526316|0.48247368421052633|
|    PR0000004|     91.4866581956798|       713|  India|0.9722222222222222|0.30223611111111115|
|    PR0000005|    86.04651162790698|       289|  India| 2.364864864864865| 0.3901013513513513|
|    PR0000006|    82.98538622129436|       745|  India| 6.289308176100629| 0.3651069182389937|
|    PR0000007|    81.97424892703863|       945|  India|1.0471204188481675|0.37231413612565445|
|    PR0000008|   103.82830626450117|   

dropping data


In [71]:
df_no_sup = df.drop("supervisor")

In [72]:
df_drop_multi = df.drop("supervisor", "maintenance_flag")

In [73]:
df_dedup = df.dropDuplicates()

In [74]:
df_unique_prod = df.dropDuplicates(["production_id"])

In [75]:
df_no_null_op = df.na.drop(subset=["operator_id"])

In [78]:
df_no_temp_vib = df.na.drop(how="all", subset=["temperature_c", "vibration_mm_s"])

In [79]:
df.select("product_family").distinct().show()

+--------------+
|product_family|
+--------------+
|         Pumps|
|        Valves|
|   Controllers|
|        Motors|
+--------------+



In [82]:
df.select("plant_id", "machine_id").distinct().show()

+--------+----------+
|plant_id|machine_id|
+--------+----------+
|    PL01|     MC001|
|    PL01|     MC004|
|    PL04|     MC019|
|    PL04|     MC017|
|    PL05|     MC022|
|    PL04|     MC018|
|    PL01|     MC003|
|    PL06|     MC029|
|    PL02|     MC007|
|    PL03|     MC013|
|    PL06|     MC030|
|    PL04|     MC020|
|    PL05|     MC021|
|    PL01|     MC005|
|    PL02|     MC010|
|    PL04|     MC016|
|    PL06|     MC027|
|    PL02|     MC006|
|    PL01|     MC002|
|    PL06|     MC028|
+--------+----------+
only showing top 20 rows


sorting

In [83]:
df.orderBy("units_produced", ascending=True).show(10)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0038643|     2026-08-06|    PL02|     MC009|     OP0361|Evening|     MTR-100|        Motors|          250|           205|              6|              73|     65.18|        55.99|          2.72|          Pass|              No|    Harini|
|    PR0059668|     

In [84]:
df.orderBy(col("units_produced").desc()).show(10)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0087560|     2026-06-29|    PL04|     MC018|     OP0587|  Night|     VLV-120|        Valves|         1199|          1245|             60|               0|    500.87|        58.11|          3.58|          Pass|             Yes|    Charan|
|    PR0008535|     

In [85]:
df.orderBy(col("defective_units").desc()).show(10)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0051359|     2026-09-16|    PL02|     MC009|     OP0181|Morning|     CTL-230|   Controllers|         1192|          1230|            108|               8|    555.34|        71.97|          2.09|        Rework|              No|     Irfan|
|    PR0108009|     

In [87]:
df.orderBy(col("plant_id").asc(), col("units_produced").desc()).show(10)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0073059|     2026-04-01|    PL01|     MC003|     OP0110|Evening|     MTR-200|        Motors|         1198|          1239|             11|               0|     282.6|        50.32|          3.91|        Rework|              No|     Gopal|
|    PR0002678|     

In [88]:
df.orderBy(col("units_produced").desc()).show(20)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0087560|     2026-06-29|    PL04|     MC018|     OP0587|  Night|     VLV-120|        Valves|         1199|          1245|             60|               0|    500.87|        58.11|          3.58|          Pass|             Yes|    Charan|
|    PR0008535|     

In [89]:
df.orderBy(col("downtime_minutes").desc()).show(10)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0092818|     2026-02-16|    PL06|     MC027|     OP0016|Morning|     PMP-210|         Pumps|         1066|           907|              2|             281|    359.48|        54.03|           2.8|          Pass|              No|     Farah|
|    PR0045363|     

In [91]:
df.orderBy(col("vibration_mm_s").desc()).select("machine_id", "vibration_mm_s").show(10)

+----------+--------------+
|machine_id|vibration_mm_s|
+----------+--------------+
|     MC012|          8.56|
|     MC030|          8.25|
|     MC027|          7.85|
|     MC010|           7.8|
|     MC018|          7.79|
|     MC024|          7.77|
|     MC015|          7.76|
|     MC027|          7.75|
|     MC015|          7.67|
|     MC016|          7.63|
+----------+--------------+
only showing top 10 rows


8.aggregations

In [92]:
from pyspark.sql.functions import sum, avg, max, min

In [94]:
from pyspark.sql.functions import sum, avg, max, min

df.agg(
    sum("units_planned").alias("total_units_planned"),
    sum("units_produced").alias("total_units_produced"),
    sum("defective_units").alias("total_defective_units"),
    avg("units_produced").alias("avg_units_produced"),
    max("units_produced").alias("max_units_produced"),
    min("units_produced").alias("min_units_produced"),
    avg("downtime_minutes").alias("avg_downtime"),
    sum("energy_kwh").alias("total_energy"),
    avg("temperature_c").alias("avg_temperature"),
    avg("vibration_mm_s").alias("avg_vibration")
).show()

+-------------------+--------------------+---------------------+------------------+------------------+------------------+-----------------+--------------------+------------------+-----------------+
|total_units_planned|total_units_produced|total_defective_units|avg_units_produced|max_units_produced|min_units_produced|     avg_downtime|        total_energy|   avg_temperature|    avg_vibration|
+-------------------+--------------------+---------------------+------------------+------------------+------------------+-----------------+--------------------+------------------+-----------------+
|           87094621|            80927888|              2030244| 674.3990666666666|              1245|               205|33.93149474428351|2.8368014080000117E7|63.021532392167266|3.398737691604917|
+-------------------+--------------------+---------------------+------------------+------------------+------------------+-----------------+--------------------+------------------+-----------------+



9.groupby

In [96]:
df.groupBy("plant_id").agg(sum("units_produced").alias("total_production")).orderBy(col("total_production").desc()).show()

+--------+----------------+
|plant_id|total_production|
+--------+----------------+
|    PL03|        13547769|
|    PL02|        13546452|
|    PL04|        13529935|
|    PL06|        13492337|
|    PL01|        13488232|
|    PL05|        13323163|
+--------+----------------+



In [97]:
df.groupBy("product_family").agg(sum("units_produced").alias("total_production")).orderBy(col("total_production").desc()).show()

+--------------+----------------+
|product_family|total_production|
+--------------+----------------+
|        Valves|        20433082|
|         Pumps|        20208739|
|   Controllers|        20168572|
|        Motors|        20117495|
+--------------+----------------+



In [98]:
df.groupBy("plant_id").agg(sum("defective_units").alias("total_defects")).orderBy(col("total_defects").desc()).show()

+--------+-------------+
|plant_id|total_defects|
+--------+-------------+
|    PL01|       339720|
|    PL02|       339647|
|    PL03|       339264|
|    PL06|       338993|
|    PL04|       338118|
|    PL05|       334502|
+--------+-------------+



In [99]:
df.groupBy("shift").agg(avg("downtime_minutes").alias("avg_downtime")).orderBy(col("avg_downtime").desc()).show()

+-------+------------------+
|  shift|      avg_downtime|
+-------+------------------+
|Evening|  34.2213017151833|
|  Night|33.953058820580935|
|Morning| 33.61973080530278|
+-------+------------------+



In [100]:
df.groupBy("product_family").agg(avg("energy_kwh").alias("avg_energy")).orderBy(col("avg_energy").desc()).show()

+--------------+------------------+
|product_family|        avg_energy|
+--------------+------------------+
|        Motors|237.21818224597516|
|         Pumps|236.89137927582252|
|        Valves|236.34508750495314|
|   Controllers|235.15370433333288|
+--------------+------------------+



In [101]:
df.groupBy("quality_status").count().orderBy(col("count").desc()).show()

+--------------+-----+
|quality_status|count|
+--------------+-----+
|          Pass|74328|
|        Rework|22928|
|        Reject|22744|
+--------------+-----+



In [103]:
df.groupBy("plant_id", "shift").agg(sum("units_produced").alias("total_prod")).show()

+--------+-------+----------+
|plant_id|  shift|total_prod|
+--------+-------+----------+
|    PL02|Morning|   4561068|
|    PL03|  Night|   4599608|
|    PL04|  Night|   4492219|
|    PL01|Morning|   4447851|
|    PL04|Evening|   4540617|
|    PL05|Morning|   4431446|
|    PL02|  Night|   4558467|
|    PL01|  Night|   4562612|
|    PL05|Evening|   4475101|
|    PL05|  Night|   4416616|
|    PL03|Morning|   4460950|
|    PL03|Evening|   4487211|
|    PL06|Evening|   4458798|
|    PL06|  Night|   4604432|
|    PL01|Evening|   4477769|
|    PL02|Evening|   4426917|
|    PL06|Morning|   4429107|
|    PL04|Morning|   4497099|
+--------+-------+----------+



In [104]:
df.groupBy("plant_id", "product_family").agg(sum("defective_units").alias("total_defects")).show()

+--------+--------------+-------------+
|plant_id|product_family|total_defects|
+--------+--------------+-------------+
|    PL06|   Controllers|        86140|
|    PL03|        Valves|        85333|
|    PL02|         Pumps|        84491|
|    PL05|         Pumps|        83607|
|    PL03|         Pumps|        84803|
|    PL01|        Motors|        84174|
|    PL05|        Valves|        82975|
|    PL05|   Controllers|        83589|
|    PL04|         Pumps|        83976|
|    PL04|        Valves|        87084|
|    PL02|   Controllers|        87120|
|    PL02|        Valves|        83476|
|    PL01|   Controllers|        83569|
|    PL01|         Pumps|        87103|
|    PL06|         Pumps|        83884|
|    PL06|        Motors|        83399|
|    PL04|   Controllers|        83161|
|    PL02|        Motors|        84560|
|    PL03|        Motors|        84926|
|    PL05|        Motors|        84331|
+--------+--------------+-------------+
only showing top 20 rows


In [106]:
df.groupBy("machine_id").agg(avg("temperature_c").alias("avg_temp")).orderBy(col("avg_temp").desc()).show()

+----------+------------------+
|machine_id|          avg_temp|
+----------+------------------+
|     MC019|63.266223444138944|
|     MC020| 63.22489208633087|
|     MC029| 63.13119731057668|
|     MC009|  63.1252779855396|
|     MC014|63.114309957574186|
|     MC023|63.109051746112684|
|     MC027|63.099668282725126|
|     MC022| 63.09403431245128|
|     MC015| 63.09026488395557|
|     MC001|  63.0778733146782|
|     MC017| 63.07674442190667|
|     MC011|63.060230414746606|
|     MC007|   63.037661716587|
|     MC016| 63.03639870582378|
|     MC026| 63.02672521957329|
|     MC021| 63.01464069491981|
|     MC005|62.979236446933044|
|     MC028| 62.97779683034558|
|     MC030| 62.96733774497327|
|     MC024| 62.96457946271644|
+----------+------------------+
only showing top 20 rows


In [107]:
df.groupBy("supervisor").agg(sum("units_produced").alias("total_output")).orderBy(col("total_output").desc()).show()

+----------+------------+
|supervisor|total_output|
+----------+------------+
|     Farah|    10276986|
|    Harini|    10215803|
|     Deepa|    10130817|
|    Bharat|    10122430|
|     Anita|    10089966|
|    Charan|    10037116|
|     Gopal|    10031531|
|     Irfan|    10023239|
+----------+------------+



10.joinng dataframes

In [108]:
import pandas as pd
from google.colab import files

# Upload your files
uploaded = files.upload()

Saving plant_master.csv to plant_master.csv
Saving machine_master.csv to machine_master.csv


In [109]:
plant_df = spark.read.csv("plant_master.csv", header=True, inferSchema=True)
machine_df = spark.read.csv("machine_master.csv", header=True, inferSchema=True)

In [112]:
df.join(plant_df, df.plant_id == plant_df.plant_id, "inner").show(5)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+--------+---------------+---------+------+-----------+-------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|plant_id|     plant_name|     city|region|year_opened|plant_manager|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+--------+---------------+---------+------+-----------+-------------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP027

In [113]:
df.join(machine_df, df.machine_id == machine_df.machine_id, "inner").show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+----------+--------+------------+-----------------+--------------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|machine_id|plant_id|machine_type|capacity_per_hour|installed_year|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+----------+--------+------------+-----------------+--------------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Even

In [118]:
full_df=df.join(plant_df, "plant_id", "inner").join(machine_df, "machine_id", "inner")

In [119]:

full_df.groupBy("city").agg(sum("units_produced").alias("total_prod")).show()

+---------+----------+
|     city|total_prod|
+---------+----------+
|  Chennai|  13547769|
|Ahmedabad|  13323163|
|  Kolkata|  13492337|
|    Noida|  13529935|
|     Pune|  13546452|
|Hyderabad|  13488232|
+---------+----------+



In [120]:
full_df.groupBy("machine_type").agg(avg("downtime_minutes").alias("avg_downtime")).show()

+------------+-----------------+
|machine_type|     avg_downtime|
+------------+-----------------+
|         CNC|33.64548129466162|
|     Testing|33.87585594724829|
|     Welding|34.00449382099613|
|    Assembly|33.96718284443325|
|     Casting| 34.1660783396419|
+------------+-----------------+



In [121]:
full_df.groupBy("region").agg(sum("defective_units").alias("total_defects")).show()

+------+-------------+
|region|total_defects|
+------+-------------+
| South|       678984|
|  East|       338993|
|  West|       674149|
| North|       338118|
+------+-------------+



In [126]:
unmatched_df = df.join(machine_df, "machine_id", "left_anti")
unmatched_df.show(5)

+----------+-------------+---------------+--------+-----------+-----+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|machine_id|production_id|production_date|plant_id|operator_id|shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+----------+-------------+---------------+--------+-----------+-----+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
+----------+-------------+---------------+--------+-----------+-----+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+



In [124]:
full_df.groupBy("plant_name").agg(sum("energy_kwh").alias("total_energy")).show()

+---------------+-----------------+
|     plant_name|     total_energy|
+---------------+-----------------+
|    Noida Plant|4746238.580000002|
|  Kolkata Plant|4723180.160000009|
|  Chennai Plant|4742748.169999995|
|     Pune Plant|4757984.949999988|
|Hyderabad Plant|4725471.189999996|
|Ahmedabad Plant| 4672391.03000001|
+---------------+-----------------+



11.updating data

In [127]:
from pyspark.sql.functions import when

In [128]:
df_updated = df.withColumn("quality_status", when(col("quality_status") == "Reject", "Failed").otherwise(col("quality_status")))

In [129]:
df_updated = df_updated.withColumn("quality_status", when(col("quality_status") == "Rework", "Requires Review").otherwise(col("quality_status")))

In [130]:
df_updated = df_updated.withColumn("units_planned", when(col("product_family") == "Motors", col("units_planned") * 1.05).otherwise(col("units_planned")))

In [131]:
df_updated = df_updated.withColumn("energy_kwh", when(col("shift") == "Night", col("energy_kwh") * 1.03).otherwise(col("energy_kwh")))

In [132]:
df_updated = df_updated.fillna({"operator_id": "UNASSIGNED"})

In [133]:
df_updated = df_updated.withColumn("supervisor", when(col("supervisor") == "Anita", "Anita Rao").otherwise(col("supervisor")))

In [135]:
df_updated = df_updated.withColumn("corrected_temperature", when(col("temperature_c") > 100, 100).otherwise(col("temperature_c")))

12. Null Values and na Functions

In [137]:
df.select(
    sum(col("operator_id").isNull().cast("int")).alias("missing_operators"),
    sum(col("temperature_c").isNull().cast("int")).alias("missing_temp"),
    sum(col("vibration_mm_s").isNull().cast("int")).alias("missing_vibration"),
    sum(col("downtime_minutes").isNull().cast("int")).alias("missing_downtime")
).show()

+-----------------+------------+-----------------+----------------+
|missing_operators|missing_temp|missing_vibration|missing_downtime|
+-----------------+------------+-----------------+----------------+
|             1524|        2132|             1788|            1177|
+-----------------+------------+-----------------+----------------+



In [138]:
df_dropped = df.na.drop()

In [139]:
df_drop_temp = df.na.drop(subset=["temperature_c"])

In [140]:
df_fill_downtime = df.na.fill(0, subset=["downtime_minutes"])

In [141]:
df_fill_op = df.na.fill("UNKNOWN", subset=["operator_id"])

In [143]:
avg_temp = df.select(avg("temperature_c")).collect()[0][0]
df_fill_temp = df.na.fill(avg_temp, subset=["temperature_c"])

In [144]:
print("Before:", df.count(), "After:", df_fill_temp.count())

Before: 120000 After: 120000


13. Built-in Functions

In [145]:
from pyspark.sql.functions import upper, lower, trim, length, concat, concat_ws, round, abs, regexp_replace, substring, coalesce

In [148]:
df_funcs = df \
    .withColumn("product_family_upper", upper(col("product_family"))) \
    .withColumn("supervisor_lower", lower(col("supervisor"))) \
    .withColumn("operator_trimmed", trim(col("operator_id"))) \
    .withColumn("product_code_length", length(col("product_code"))) \
    .withColumn("plant_machine", concat_ws("-", col("plant_id"), col("machine_id"))) \
    .withColumn("temp_rounded", round(col("temperature_c"), 1)) \
    .withColumn("unit_diff", abs(col("units_produced") - col("units_planned"))) \
    .withColumn("clean_prod_id", regexp_replace(col("production_id"), "PRD", "ID")) \
    .withColumn("prod_code_sub", substring(col("product_code"), 1, 3)) \
    .withColumn("temp_coalesce", coalesce(col("temperature_c"), lit(25.0)))

In [149]:
df_funcs.select("product_family_upper", "supervisor_lower", "temp_rounded", "unit_diff").show(5)

+--------------------+----------------+------------+---------+
|product_family_upper|supervisor_lower|temp_rounded|unit_diff|
+--------------------+----------------+------------+---------+
|              MOTORS|          charan|        61.4|       67|
|         CONTROLLERS|           farah|        64.4|       27|
|              MOTORS|           deepa|        54.1|       62|
|               PUMPS|           farah|        64.5|       67|
|               PUMPS|           deepa|        72.6|       48|
+--------------------+----------------+------------+---------+
only showing top 5 rows


when() / otherwise()

In [151]:
df_categorized = df \
    .withColumn("efficiency_category",
        when((col("units_produced") / col("units_planned")) * 100 >= 100, "Above Target")
        .when((col("units_produced") / col("units_planned")) * 100 >= 95, "On Target")
        .when((col("units_produced") / col("units_planned")) * 100 >= 85, "Below Target")
        .otherwise("Critical")
    ) \
    .withColumn("defect_category",
        when(col("defective_units") == 0, "Perfect")
        .when(col("defective_units").between(1, 10), "Normal")
        .when(col("defective_units").between(11, 25), "Attention")
        .otherwise("High Defect")
    ) \
    .withColumn("machine_condition",
        when((col("temperature_c") > 80) | (col("vibration_mm_s") > 5.0), "Inspect")
        .otherwise("Normal")
    )

In [152]:
df_categorized.groupBy("efficiency_category").count().show()
df_categorized.groupBy("defect_category").count().show()
df_categorized.groupBy("machine_condition").count().show()

+-------------------+-----+
|efficiency_category|count|
+-------------------+-----+
|       Above Target|21735|
|          On Target|26825|
|       Below Target|54619|
|           Critical|16821|
+-------------------+-----+

+---------------+-----+
|defect_category|count|
+---------------+-----+
|      Attention|45745|
|        Perfect|11491|
|    High Defect|28058|
|         Normal|34706|
+---------------+-----+

+-----------------+------+
|machine_condition| count|
+-----------------+------+
|          Inspect| 10886|
|           Normal|109114|
+-----------------+------+



15. Complex JSON

In [153]:
from google.colab import files

# Upload your files
uploaded = files.upload()

Saving machine_configurations.json to machine_configurations.json


In [155]:
machine_json = spark.read.option("multiline", "true").json("machine_configurations.json")

In [156]:
machine_json.printSchema()

root
 |-- capacity_per_hour: long (nullable = true)
 |-- installed_year: long (nullable = true)
 |-- machine_id: string (nullable = true)
 |-- machine_type: string (nullable = true)
 |-- maintenance: struct (nullable = true)
 |    |-- frequency_days: long (nullable = true)
 |    |-- last_service: string (nullable = true)
 |-- plant_id: string (nullable = true)
 |-- sensors: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- active: boolean (nullable = true)
 |    |    |-- sensor: string (nullable = true)
 |    |    |-- unit: string (nullable = true)
 |-- specifications: struct (nullable = true)
 |    |-- max_temperature_c: long (nullable = true)
 |    |-- rated_power_kw: long (nullable = true)
 |    |-- vibration_limit_mm_s: double (nullable = true)



In [157]:
machine_json.select("machine_id", "machine_type").show()

+----------+------------+
|machine_id|machine_type|
+----------+------------+
|     MC001|         CNC|
|     MC002|    Assembly|
|     MC003|     Casting|
|     MC004|     Welding|
|     MC005|     Testing|
|     MC006|         CNC|
|     MC007|    Assembly|
|     MC008|     Casting|
|     MC009|     Welding|
|     MC010|     Testing|
|     MC011|         CNC|
|     MC012|    Assembly|
|     MC013|     Casting|
|     MC014|     Welding|
|     MC015|     Testing|
|     MC016|         CNC|
|     MC017|    Assembly|
|     MC018|     Casting|
|     MC019|     Welding|
|     MC020|     Testing|
+----------+------------+
only showing top 20 rows


In [159]:
machine_json.select("machine_id", "specifications.rated_power_kw", "specifications.max_temperature_c", "specifications.vibration_limit_mm_s").show()

+----------+--------------+-----------------+--------------------+
|machine_id|rated_power_kw|max_temperature_c|vibration_limit_mm_s|
+----------+--------------+-----------------+--------------------+
|     MC001|            22|               85|                 5.0|
|     MC002|            22|               85|                 5.5|
|     MC003|            55|               90|                 4.5|
|     MC004|            22|               85|                 4.5|
|     MC005|            30|               85|                 4.5|
|     MC006|            30|               85|                 5.5|
|     MC007|            22|               80|                 4.5|
|     MC008|            55|               75|                 5.0|
|     MC009|            30|               85|                 5.5|
|     MC010|            55|               90|                 4.5|
|     MC011|            30|               90|                 6.0|
|     MC012|            55|               80|                 

In [162]:
from pyspark.sql.functions import explode
exploded_json = machine_json.withColumn("sensor", explode("sensors"))
exploded_json.select("machine_id", "sensor.sensor", "sensor.active").show(10, truncate=False)

+----------+-----------+------+
|machine_id|sensor     |active|
+----------+-----------+------+
|MC001     |temperature|true  |
|MC001     |vibration  |true  |
|MC001     |energy     |true  |
|MC002     |temperature|true  |
|MC002     |vibration  |true  |
|MC002     |energy     |true  |
|MC003     |temperature|true  |
|MC003     |vibration  |true  |
|MC003     |energy     |true  |
|MC004     |temperature|true  |
+----------+-----------+------+
only showing top 10 rows


16. Union and Union ALL

In [163]:
from google.colab import files

# Upload your files
uploaded = files.upload()

Saving manufacturing_production_extra_12k.csv to manufacturing_production_extra_12k.csv


In [164]:
extra_df = spark.read.csv("manufacturing_production_extra_12k.csv", header=True, inferSchema=True)

In [165]:
print("Main count:", df.count())
print("Extra count:", extra_df.count())

Main count: 120000
Extra count: 12500


In [166]:
union_df = df.union(extra_df)

In [167]:
print("Union count:", union_df.count())

Union count: 132500


In [168]:
dedup_union = union_df.dropDuplicates(["production_id"])

In [169]:
union_by_name_df = df.unionByName(extra_df, allowMissingColumns=True)

17. Window Functions

In [170]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, rank, dense_rank, sum, avg, lag, lead

In [171]:
plant_window = Window.partitionBy("plant").orderBy(col("units_produced").desc())
machine_window = Window.partitionBy("plant", "machine").orderBy("production_date")

In [173]:
plant_window = Window.partitionBy("plant_id").orderBy(col("units_produced").desc())
machine_window = Window.partitionBy("plant_id", "machine_id").orderBy("production_date")

df_windowed = df \
    .withColumn("row_num", row_number().over(plant_window)) \
    .withColumn("rank_val", rank().over(plant_window)) \
    .withColumn("dense_rank_val", dense_rank().over(plant_window)) \
    .withColumn("total_plant_prod", sum("units_produced").over(Window.partitionBy("plant_id"))) \
    .withColumn("avg_machine_prod", avg("units_produced").over(Window.partitionBy("machine_id"))) \
    .withColumn("prev_output", lag("units_produced", 1).over(machine_window)) \
    .withColumn("next_output", lead("units_produced", 1).over(machine_window)) \
    .withColumn("output_change", col("units_produced") - lag("units_produced", 1).over(machine_window))

df_windowed.select("production_id", "plant_id", "machine_id", "units_produced", "row_num", "total_plant_prod", "prev_output", "output_change").show(10)

+-------------+--------+----------+--------------+-------+----------------+-----------+-------------+
|production_id|plant_id|machine_id|units_produced|row_num|total_plant_prod|prev_output|output_change|
+-------------+--------+----------+--------------+-------+----------------+-----------+-------------+
|    PR0009794|    PL02|     MC006|          1046|   1604|        13546452|       NULL|         NULL|
|    PR0062624|    PL02|     MC006|          1020|   2081|        13546452|       1046|          -26|
|    PR0084920|    PL02|     MC006|          1014|   2209|        13546452|       1020|           -6|
|    PR0116519|    PL02|     MC006|           968|   3253|        13546452|       1014|          -46|
|    PR0068042|    PL02|     MC006|           872|   5431|        13546452|        968|          -96|
|    PR0056314|    PL02|     MC006|           829|   6435|        13546452|        872|          -43|
|    PR0057998|    PL02|     MC006|           546|  12894|        13546452|       

In [174]:
from pyspark.sql.functions import to_date, year, month, dayofmonth, date_format, date_add, datediff, lit

In [175]:
df_dates = df \
    .withColumn("prod_date", to_date(col("production_date"), "yyyy-MM-dd")) \
    .withColumn("prod_year", year(col("prod_date"))) \
    .withColumn("prod_month", month(col("prod_date"))) \
    .withColumn("prod_day", dayofmonth(col("prod_date"))) \
    .withColumn("month_name", date_format(col("prod_date"), "MMMM")) \
    .withColumn("day_name", date_format(col("prod_date"), "EEEE")) \
    .withColumn("year_month", date_format(col("prod_date"), "yyyy-MM")) \
    .withColumn("date_plus_7", date_add(col("prod_date"), 7))

In [176]:
df_dates.groupBy("year_month").agg(sum("units_produced").alias("monthly_production")).orderBy("year_month").show()

+----------+------------------+
|year_month|monthly_production|
+----------+------------------+
|   2026-01|           8478059|
|   2026-02|           7810891|
|   2026-03|           8441121|
|   2026-04|           8211005|
|   2026-05|           8507542|
|   2026-06|           8400505|
|   2026-07|           8476416|
|   2026-08|           8529150|
|   2026-09|           8115221|
|   2026-10|           5957978|
+----------+------------------+

